# DBSCAN 密度聚类

## 目标

理解 DBSCAN（基于密度的空间聚类）算法的工作原理。

- 理解 DBSCAN 的核心概念
- 学习参数调优方法
- 与 K-Means、层次聚类对比
- 处理噪声数据的能力
- 发现任意形状的簇

## 1. 数据准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
from sklearn.datasets import make_blobs, make_moons, make_circles, make_swiss_roll
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
import seaborn as sns

np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 生成合成数据
def generate_dbscan_datasets():
    datasets = {}
    
    # 月形数据
    X_moons, y_moons = make_moons(
        n_samples=300, noise=0.1, random_state=42
    )
    datasets['moons'] = (X_moons, y_moons, '月形数据')
    
    # 环形数据
    X_circles, y_circles = make_circles(
        n_samples=300, factor=0.5, noise=0.05, random_state=42
    )
    datasets['circles'] = (X_circles, y_circles, '环形数据')
    
    # 带噪声的球形簇
    X_blobs, y_blobs = make_blobs(
        n_samples=300, n_features=2, centers=4,
        cluster_std=1.5, random_state=42
    )
    # 添加噪声点
    noise = np.random.uniform(-10, 10, (30, 2))
    X_blobs_noisy = np.vstack([X_blobs, noise])
    y_blobs_noisy = np.concatenate([y_blobs, [-1] * 30])
    datasets['blobs_noisy'] = (X_blobs_noisy, y_blobs_noisy, '带噪声的球形簇')
    
    # 各向异性数据
    X_aniso, y_aniso = make_blobs(
        n_samples=300, n_features=2, centers=3,
        cluster_std=1, random_state=42
    )
    transformation = [[0.6, -0.6], [-0.4, 0.8]]
    X_aniso = np.dot(X_aniso, transformation)
    datasets['aniso'] = (X_aniso, y_aniso, '各向异性数据')
    
    # 稀疏数据
    X_sparse, y_sparse = make_blobs(
        n_samples=300, n_features=2, centers=3,
        cluster_std=[0.5, 2.0, 3.0], random_state=42
    )
    datasets['sparse'] = (X_sparse, y_sparse, '稀疏数据')
    
    return datasets

datasets = generate_dbscan_datasets()

print("生成的数据集:")
for name, (X, y, desc) in datasets.items():
    print(f"  {name}: {X.shape} - {desc}")

In [ ]:
# 可视化数据集
plt.figure(figsize=(12, 5))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    plt.subplot(2, 3, i+1)
    if y is not None and -1 in y:
        # 有噪声标签
        noise_mask = (y == -1)
        plt.scatter(X[~noise_mask, 0], X[~noise_mask, 1], c=y[~noise_mask], 
                   cmap='viridis', alpha=0.6, s=30)
        plt.scatter(X[noise_mask, 0], X[noise_mask, 1], c='red', 
                   marker='x', s=50, label='噪声')
    else:
        plt.scatter(X[:, 0], X[:, 1], c=y if y is not None else 'gray', 
                   cmap='viridis', alpha=0.6, s=30)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{desc}\n({name})')
    plt.grid(True, alpha=0.3)
    if y is not None and -1 in y:
        plt.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 2. DBSCAN 基础

In [ ]:
# DBSCAN 核心概念
print("DBSCAN 核心概念:")
print("- eps (epsilon): 邻域半径，定义点的邻域大小")
print("- min_samples: 邻域内最少点数，定义核心点")
print("\n点的分类:")
print("1. 核心点: 邻域内至少有 min_samples 个点")
print("2. 边界点: 不是核心点，但在某个核心点的邻域内")
print("3. 噪声点: 既不是核心点，也不是边界点")
print("\n簇的定义:")
print("- 通过核心点的密度可达性形成簇")
print("- 噪声点不属于任何簇（标签为 -1）")

In [ ]:
# 使用 DBSCAN 聚类
X_moons, y_moons, _ = datasets['moons']

# DBSCAN 聚类
dbscan = DBSCAN(eps=0.2, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_moons)

print("DBSCAN 结果:")
print(f"  发现簇数: {len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)}")
print(f"  噪声点数: {np.sum(labels_dbscan == -1)}")
print(f"  簇大小分布: {np.bincount(labels_dbscan + 1)[1:]}")
print(f"  参数: eps={dbscan.eps}, min_samples={dbscan.min_samples}")

In [ ]:
# 可视化 DBSCAN 结果
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 原始数据
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', alpha=0.6, s=30)
axes[0].set_xlabel('特征 1')
axes[0].set_ylabel('特征 2')
axes[0].set_title('原始标签')
axes[0].grid(True, alpha=0.3)

# DBSCAN 结果（标记噪声点）
unique_labels = set(labels_dbscan)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for k, col in zip(unique_labels, colors):
    if k == -1:
        # 噪声点用黑色标记
        col = [0, 0, 0, 1]
        markersize = 6
        marker = 'x'
        label = '噪声'
    else:
        markersize = 30
        marker = 'o'
        label = f'簇 {k}'
    
    class_member_mask = (labels_dbscan == k)
    
    xy = X_moons[class_member_mask]
    axes[1].scatter(xy[:, 0], xy[:, 1], c=[col], s=markersize, 
                  marker=marker, alpha=0.6, label=label)

axes[1].set_xlabel('特征 1')
axes[1].set_ylabel('特征 2')
axes[1].set_title('DBSCAN 聚类结果')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# 计算评估指标（只考虑非噪声点）
ari = adjusted_rand_score(y_moons, labels_dbscan)
non_noise_mask = labels_dbscan != -1
if np.sum(non_noise_mask) > 1:
    silhouette = silhouette_score(X_moons[non_noise_mask], labels_dbscan[non_noise_mask])
else:
    silhouette = np.nan

print(f"\n评估指标:")
print(f"  ARI (调整兰德指数): {ari:.3f}")
print(f"  轮廓系数: {silhouette:.3f}")

## 3. 参数调优：eps 和 min_samples

In [ ]:
# 测试不同的 eps 值
eps_values = [0.05, 0.1, 0.2, 0.3, 0.5]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, eps in enumerate(eps_values):
    # 聚类
    dbscan_temp = DBSCAN(eps=eps, min_samples=5)
    labels_temp = dbscan_temp.fit_predict(X_moons)
    
    n_clusters = len(set(labels_temp)) - (1 if -1 in labels_temp else 0)
    n_noise = np.sum(labels_temp == -1)
    ari = adjusted_rand_score(y_moons, labels_temp)
    
    # 绘制聚类结果
    unique_labels = set(labels_temp)
    for k in unique_labels:
        class_member_mask = (labels_temp == k)
        if k == -1:
            xy = X_moons[class_member_mask]
            axes[0, i].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
        else:
            xy = X_moons[class_member_mask]
            axes[0, i].scatter(xy[:, 0], xy[:, 1], label=f'簇 {k}', alpha=0.6)
    
    axes[0, i].set_xlabel('特征 1')
    axes[0, i].set_ylabel('特征 2')
    axes[0, i].set_title(f'eps={eps}\n簇数={n_clusters}, 噪声={n_noise}')
    axes[0, i].grid(True, alpha=0.3)
    
    # 绘制 eps 范围示意图（中心点的邻域）
    center_point = X_moons[0]
    circle = plt.Circle(center_point, eps, color='red', fill=False, 
                     linestyle='--', alpha=0.5)
    axes[1, i].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', alpha=0.3)
    axes[1, i].scatter(center_point[0], center_point[1], c='red', s=100, marker='*', label='中心点')
    axes[1, i].add_patch(circle)
    axes[1, i].set_xlabel('特征 1')
    axes[1, i].set_ylabel('特征 2')
    axes[1, i].set_title(f'邻域半径 = {eps}')
    axes[1, i].grid(True, alpha=0.3)
    axes[1, i].legend()

plt.tight_layout()
plt.show()

print("eps 参数影响:")
print("- eps 太小: 大部分点被标记为噪声，簇数过多")
print("- eps 太大: 所有点合并成一个簇，噪声点减少")
print("- 合适的 eps: 能正确识别簇的边界和噪声点")

In [ ]:
# 测试不同的 min_samples 值
min_samples_values = [3, 5, 10, 20, 30]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, min_s in enumerate(min_samples_values):
    # 聚类
    dbscan_temp = DBSCAN(eps=0.2, min_samples=min_s)
    labels_temp = dbscan_temp.fit_predict(X_moons)
    
    n_clusters = len(set(labels_temp)) - (1 if -1 in labels_temp else 0)
    n_noise = np.sum(labels_temp == -1)
    ari = adjusted_rand_score(y_moons, labels_temp)
    
    # 绘制聚类结果
    unique_labels = set(labels_temp)
    for k in unique_labels:
        class_member_mask = (labels_temp == k)
        if k == -1:
            xy = X_moons[class_member_mask]
            axes[0, i].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
        else:
            xy = X_moons[class_member_mask]
            axes[0, i].scatter(xy[:, 0], xy[:, 1], label=f'簇 {k}', alpha=0.6)
    
    axes[0, i].set_xlabel('特征 1')
    axes[0, i].set_ylabel('特征 2')
    axes[0, i].set_title(f'min_samples={min_s}\n簇数={n_clusters}, 噪声={n_noise}')
    axes[0, i].grid(True, alpha=0.3)
    
    # 绘制邻域点数示意图
    center_point = X_moons[0]
    distances = np.linalg.norm(X_moons - center_point, axis=1)
    neighbors_mask = distances <= 0.2
    
    axes[1, i].scatter(X_moons[:, 0], X_moons[:, 1], c='gray', alpha=0.3)
    axes[1, i].scatter(X_moons[neighbors_mask, 0], X_moons[neighbors_mask, 1], 
                     c='red', s=50, label=f'邻域内\n({np.sum(neighbors_mask)}个点)')
    axes[1, i].scatter(center_point[0], center_point[1], c='black', s=100, marker='*', label='中心点')
    axes[1, i].set_xlabel('特征 1')
    axes[1, i].set_ylabel('特征 2')
    axes[1, i].set_title(f'核心点需要 >= {min_s} 个邻域点')
    axes[1, i].grid(True, alpha=0.3)
    axes[1, i].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("min_samples 参数影响:")
print("- min_samples 太小: 对噪声敏感，容易形成过多小簇")
print("- min_samples 太大: 对密度变化不敏感，可能将噪声点归入簇")
print("- 合适的 min_samples: 能有效过滤噪声，同时保持簇的完整性")
print("\n经验法则: min_samples >= 维度 + 1 (2D数据至少为3)")

## 4. 使用 K-距离图选择 eps

In [ ]:
from sklearn.neighbors import NearestNeighbors

# K-距离图方法选择 eps
def plot_k_distance_graph(X, k=5):
    """绘制 K-距离图用于选择 eps"""
    # 计算 k 个最近邻的距离
    nbrs = NearestNeighbors(n_neighbors=k).fit(X)
    distances, indices = nbrs.kneighbors(X)
    
    # 取第 k 个最近邻的距离（按升序排列）
    k_distances = distances[:, k-1]
    k_distances_sorted = np.sort(k_distances)
    
    # 绘图
    plt.figure(figsize=(10, 5))
    plt.plot(range(len(k_distances_sorted)), k_distances_sorted, 'b-', linewidth=2)
    plt.xlabel('点（按距离排序）')
    plt.ylabel(f'第 {k} 最近邻距离')
    plt.title('K-距离图：选择 eps 参数')
    plt.grid(True, alpha=0.3)
    
    # 标记可能的拐点
    # 简单方法：选择曲率最大的点
    from scipy.signal import argrelextrema
    
    # 计算差分
    diff = np.diff(k_distances_sorted)
    diff2 = np.diff(diff)
    
    # 找到肘部（简化方法）
    elbow_idx = np.argmax(diff2)
    suggested_eps = k_distances_sorted[elbow_idx]
    
    plt.axhline(y=suggested_eps, color='red', linestyle='--', 
               label=f'建议 eps = {suggested_eps:.3f}', linewidth=2)
    plt.axvline(x=elbow_idx, color='red', linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()
    
    return suggested_eps

# 为月形数据选择 eps
suggested_eps = plot_k_distance_graph(X_moons, k=5)

print(f"\nK-距离图方法:")
print(f"1. 计算每个点到其第 k 个最近邻的距离")
print(f"2. 将距离按升序排列")
print(f"3. 找到曲线的"肘部"（拐点）")
print(f"4. 拐点对应的 y 值即为建议的 eps")
print(f"\n建议的 eps 值: {suggested_eps:.3f}")

In [ ]:
# 使用建议的 eps 进行聚类
dbscan_optimized = DBSCAN(eps=suggested_eps, min_samples=5)
labels_optimized = dbscan_optimized.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 原始聚类
unique_labels = set(labels_dbscan)
for k in unique_labels:
    class_member_mask = (labels_dbscan == k)
    if k == -1:
        xy = X_moons[class_member_mask]
        axes[0].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
    else:
        xy = X_moons[class_member_mask]
        axes[0].scatter(xy[:, 0], xy[:, 1], label=f'簇 {k}', alpha=0.6)
axes[0].set_xlabel('特征 1')
axes[0].set_ylabel('特征 2')
axes[0].set_title(f'原始参数 (eps=0.2)\nARI={adjusted_rand_score(y_moons, labels_dbscan):.3f}')
axes[0].grid(True, alpha=0.3)

# 优化后聚类
unique_labels = set(labels_optimized)
for k in unique_labels:
    class_member_mask = (labels_optimized == k)
    if k == -1:
        xy = X_moons[class_member_mask]
        axes[1].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
    else:
        xy = X_moons[class_member_mask]
        axes[1].scatter(xy[:, 0], xy[:, 1], label=f'簇 {k}', alpha=0.6)
axes[1].set_xlabel('特征 1')
axes[1].set_ylabel('特征 2')
axes[1].set_title(f'优化参数 (eps={suggested_eps:.3f})\nARI={adjusted_rand_score(y_moons, labels_optimized):.3f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 在不同数据集上的表现

In [ ]:
# 在所有数据集上测试 DBSCAN
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    # 根据数据集特点调整参数
    if name == 'moons':
        dbscan_temp = DBSCAN(eps=0.2, min_samples=5)
    elif name == 'circles':
        dbscan_temp = DBSCAN(eps=0.1, min_samples=5)
    elif name == 'blobs_noisy':
        dbscan_temp = DBSCAN(eps=1.5, min_samples=5)
    elif name == 'aniso':
        dbscan_temp = DBSCAN(eps=0.5, min_samples=5)
    elif name == 'sparse':
        dbscan_temp = DBSCAN(eps=1.5, min_samples=5)
    
    labels_temp = dbscan_temp.fit_predict(X)
    
    n_clusters = len(set(labels_temp)) - (1 if -1 in labels_temp else 0)
    n_noise = np.sum(labels_temp == -1)
    ari = adjusted_rand_score(y, labels_temp) if y is not None and -1 not in y else np.nan
    
    # 绘制原始数据
    if y is not None and -1 in y:
        noise_mask = (y == -1)
        axes[0, i].scatter(X[~noise_mask, 0], X[~noise_mask, 1], 
                          c=y[~noise_mask], cmap='viridis', alpha=0.6, s=30)
        axes[0, i].scatter(X[noise_mask, 0], X[noise_mask, 1], 
                          c='red', marker='x', s=50)
    else:
        axes[0, i].scatter(X[:, 0], X[:, 1], c=y if y is not None else 'gray', 
                          cmap='viridis', alpha=0.6, s=30)
    axes[0, i].set_xlabel('特征 1')
    axes[0, i].set_ylabel('特征 2')
    axes[0, i].set_title(f'{desc}\n原始')
    axes[0, i].grid(True, alpha=0.3)
    
    # 绘制 DBSCAN 结果
    unique_labels = set(labels_temp)
    for k in unique_labels:
        class_member_mask = (labels_temp == k)
        if k == -1:
            xy = X[class_member_mask]
            axes[1, i].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
        else:
            xy = X[class_member_mask]
            axes[1, i].scatter(xy[:, 0], xy[:, 1], label=f'簇 {k}', alpha=0.6)
    
    axes[1, i].set_xlabel('特征 1')
    axes[1, i].set_ylabel('特征 2')
    axes[1, i].set_title(f'DBSCAN\n簇={n_clusters}, 噪声={n_noise}')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("DBSCAN 在不同数据集上的表现:")
print("- 月形/环形数据: DBSCAN 表现优秀，能识别非球形簇")
print("- 带噪声数据: DBSCAN 能自动识别和过滤噪声点")
print("- 各向异性数据: DBSCAN 表现较好")
print("- 稀疏数据: DBSCAN 对密度变化敏感")

## 6. DBSCAN vs K-Means vs 层次聚类

In [ ]:
# 对比三种聚类算法
comparison_datasets = ['moons', 'circles', 'blobs_noisy']

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i, dataset_name in enumerate(comparison_datasets):
    X, y, desc = datasets[dataset_name]
    
    # K-Means
    n_clusters_kmeans = 2 if dataset_name in ['moons', 'circles'] else 4
    kmeans_temp = KMeans(n_clusters=n_clusters_kmeans, random_state=42)
    labels_kmeans = kmeans_temp.fit_predict(X)
    
    # 层次聚类
    agg_temp = AgglomerativeClustering(n_clusters=n_clusters_kmeans, linkage='ward')
    labels_agg = agg_temp.fit_predict(X)
    
    # DBSCAN
    if dataset_name == 'moons':
        dbscan_temp = DBSCAN(eps=0.2, min_samples=5)
    elif dataset_name == 'circles':
        dbscan_temp = DBSCAN(eps=0.1, min_samples=5)
    else:
        dbscan_temp = DBSCAN(eps=1.5, min_samples=5)
    labels_dbscan_temp = dbscan_temp.fit_predict(X)
    
    # 绘制 K-Means
    axes[i, 0].scatter(X[:, 0], X[:, 1], c=labels_kmeans, cmap='viridis', alpha=0.6)
    axes[i, 0].scatter(kmeans_temp.cluster_centers_[:, 0], kmeans_temp.cluster_centers_[:, 1],
                   c='black', marker='x', s=100, linewidths=2)
    axes[i, 0].set_ylabel(desc)
    if i == 0:
        axes[i, 0].set_title('K-Means')
    axes[i, 0].grid(True, alpha=0.3)
    
    # 绘制层次聚类
    axes[i, 1].scatter(X[:, 0], X[:, 1], c=labels_agg, cmap='viridis', alpha=0.6)
    if i == 0:
        axes[i, 1].set_title('层次聚类')
    axes[i, 1].grid(True, alpha=0.3)
    
    # 绘制 DBSCAN
    unique_labels = set(labels_dbscan_temp)
    for k in unique_labels:
        class_member_mask = (labels_dbscan_temp == k)
        if k == -1:
            xy = X[class_member_mask]
            axes[i, 2].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
        else:
            xy = X[class_member_mask]
            axes[i, 2].scatter(xy[:, 0], xy[:, 1], alpha=0.6)
    if i == 0:
        axes[i, 2].set_title('DBSCAN')
    axes[i, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("三种聚类算法对比:")
print("| 特点 | K-Means | 层次聚类 | DBSCAN |")
print("|------|---------|----------|----------|")
print("| 需要指定簇数 | 必须 | 可选 | 自动 |")
print("| 处理噪声 | 否 | 否 | 是 |")
print("| 任意形状 | 否 | 部分可以 | 是 |")
print("| 参数 | K | 链接方式 | eps, min_samples |")
print("| 计算复杂度 | O(nkt) | O(n²) | O(n log n) |")
print("| 大数据 | 适合 | 不适合 | 适合 |")
print("| 密度变化敏感 | 否 | 否 | 是 |")

## 7. DBSCAN 的优势：处理噪声数据

In [ ]:
# 演示 DBSCAN 处理噪声的能力
X_blobs_noisy, y_blobs_noisy, _ = datasets['blobs_noisy']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# DBSCAN
dbscan_noise = DBSCAN(eps=1.5, min_samples=5)
labels_dbscan_noise = dbscan_noise.fit_predict(X_blobs_noisy)

unique_labels = set(labels_dbscan_noise)
for k in unique_labels:
    class_member_mask = (labels_dbscan_noise == k)
    if k == -1:
        xy = X_blobs_noisy[class_member_mask]
        axes[0, 0].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=50, alpha=0.8, linewidths=2)
    else:
        xy = X_blobs_noisy[class_member_mask]
        axes[0, 0].scatter(xy[:, 0], xy[:, 1], label=f'簇 {k}', alpha=0.6)
axes[0, 0].set_xlabel('特征 1')
axes[0, 0].set_ylabel('特征 2')
axes[0, 0].set_title(f'DBSCAN\n检测到 {np.sum(labels_dbscan_noise == -1)} 个噪声点')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# K-Means
kmeans_noise = KMeans(n_clusters=4, random_state=42)
labels_kmeans_noise = kmeans_noise.fit_predict(X_blobs_noisy)

axes[0, 1].scatter(X_blobs_noisy[:, 0], X_blobs_noisy[:, 1], c=labels_kmeans_noise, 
                   cmap='viridis', alpha=0.6)
axes[0, 1].scatter(kmeans_noise.cluster_centers_[:, 0], kmeans_noise.cluster_centers_[:, 1],
                   c='black', marker='x', s=100, linewidths=2)
axes[0, 1].set_xlabel('特征 1')
axes[0, 1].set_ylabel('特征 2')
axes[0, 1].set_title('K-Means\n无法识别噪声')
axes[0, 1].grid(True, alpha=0.3)

# 层次聚类
agg_noise = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_agg_noise = agg_noise.fit_predict(X_blobs_noisy)

axes[0, 2].scatter(X_blobs_noisy[:, 0], X_blobs_noisy[:, 1], c=labels_agg_noise, 
                   cmap='viridis', alpha=0.6)
axes[0, 2].set_xlabel('特征 1')
axes[0, 2].set_ylabel('特征 2')
axes[0, 2].set_title('层次聚类\n无法识别噪声')
axes[0, 2].grid(True, alpha=0.3)

# 噪声点分析
noise_mask = (labels_dbscan_noise == -1)
noise_points = X_blobs_noisy[noise_mask]

axes[1, 0].scatter(noise_points[:, 0], noise_points[:, 1], c='red', marker='x', s=100, 
                   alpha=0.8, linewidths=2)
axes[1, 0].set_xlabel('特征 1')
axes[1, 0].set_ylabel('特征 2')
axes[1, 0].set_title('DBSCAN 识别的噪声点')
axes[1, 0].grid(True, alpha=0.3)

# 到最近簇中心的距离
from scipy.spatial.distance import cdist
distances_to_centers = cdist(noise_points, kmeans_noise.cluster_centers_).min(axis=1)
axes[1, 1].hist(distances_to_centers, bins=10, edgecolor='black')
axes[1, 1].set_xlabel('到最近簇中心的距离')
axes[1, 1].set_ylabel('噪声点数量')
axes[1, 1].set_title('噪声点分布')
axes[1, 1].grid(True, alpha=0.3)

# 对比分析
analysis_text = f"""DBSCAN 噪声处理能力:

检测到的噪声点: {np.sum(labels_dbscan_noise == -1)}
实际噪声点: {np.sum(y_blobs_noisy == -1)}

K-Means 问题:
- 噪声点拉偏质心
- 无法识别异常值
- 所有点都必须属于某个簇

DBSCAN 优势:
- 自动识别噪声点
- 不受异常值影响
- 能发现任意形状的簇

层次聚类:
- 类似 K-Means，所有点都属于某个簇
- 无法区分噪声和正常点
"""
axes[1, 2].text(0.1, 0.1, analysis_text, fontsize=10, verticalalignment='bottom',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 8. OPTICS: 自动参数选择的 DBSCAN 变体

In [ ]:
from sklearn.cluster import OPTICS

# OPTICS 聚类（自动调整 eps）
optics = OPTICS(min_samples=5, xi=0.05)
labels_optics = optics.fit_predict(X_moons)

print("OPTICS 结果:")
print(f"  发现簇数: {len(set(labels_optics)) - (1 if -1 in labels_optics else 0)}")
print(f"  噪声点数: {np.sum(labels_optics == -1)}")
print(f"  参数: min_samples={optics.min_samples}, xi={optics.xi}")
print("\nOPTICS 特点:")
print("- 不需要预先指定 eps")
print("- 自动检测不同密度的簇")
print("- 计算复杂度较高")
print("- 适合密度变化大的数据")

In [ ]:
# 可视化 OPTICS 结果
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始数据
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', alpha=0.6)
axes[0].set_xlabel('特征 1')
axes[0].set_ylabel('特征 2')
axes[0].set_title('原始标签')
axes[0].grid(True, alpha=0.3)

# DBSCAN 结果
unique_labels = set(labels_dbscan)
for k in unique_labels:
    class_member_mask = (labels_dbscan == k)
    if k == -1:
        xy = X_moons[class_member_mask]
        axes[1].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
    else:
        xy = X_moons[class_member_mask]
        axes[1].scatter(xy[:, 0], xy[:, 1], alpha=0.6)
axes[1].set_xlabel('特征 1')
axes[1].set_ylabel('特征 2')
axes[1].set_title(f'DBSCAN (eps=0.2)')
axes[1].grid(True, alpha=0.3)

# OPTICS 结果
unique_labels = set(labels_optics)
for k in unique_labels:
    class_member_mask = (labels_optics == k)
    if k == -1:
        xy = X_moons[class_member_mask]
        axes[2].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
    else:
        xy = X_moons[class_member_mask]
        axes[2].scatter(xy[:, 0], xy[:, 1], alpha=0.6)
axes[2].set_xlabel('特征 1')
axes[2].set_ylabel('特征 2')
axes[2].set_title('OPTICS (自动 eps)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 计算评估指标
ari_dbscan = adjusted_rand_score(y_moons, labels_dbscan)
ari_optics = adjusted_rand_score(y_moons, labels_optics)

print(f"\n对比结果:")
print(f"  DBSCAN ARI: {ari_dbscan:.3f}")
print(f"  OPTICS ARI: {ari_optics:.3f}")

## 9. 实际应用：图像分割

In [ ]:
# 使用 DBSCAN 进行图像分割
from sklearn.datasets import load_sample_image
from skimage.color import rgb2gray

# 加载示例图像
try:
    image = load_sample_image('china.jpg')
except:
    # 如果无法加载，创建一个简单的测试图像
    print("无法加载示例图像，创建测试图像...")
    image = np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8)

# 显示原始图像
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title('原始图像')
plt.axis('off')

# 转换为像素点
h, w, d = image.shape
image_array = np.reshape(image, (h * w, d))
image_array = image_array / 255.0  # 归一化

# 使用 DBSCAN 进行颜色聚类
print(f"图像尺寸: {h}x{w}")
print(f"总像素数: {h * w}")
print("正在使用 DBSCAN 进行颜色聚类...")

# 使用较小的数据集进行演示
if h * w > 10000:
    # 随机采样
    sample_indices = np.random.choice(h * w, 10000, replace=False)
    image_sampled = image_array[sample_indices]
    print(f"采样 {len(sample_indices)} 个像素点")
else:
    image_sampled = image_array

# DBSCAN 聚类
dbscan_image = DBSCAN(eps=0.1, min_samples=10)
labels_image = dbscan_image.fit_predict(image_sampled)

n_colors = len(set(labels_image)) - (1 if -1 in labels_image else 0)
print(f"发现 {n_colors} 种主要颜色")

# 计算每种颜色的代表色
colors = []
for k in range(n_colors):
    mask = (labels_image == k)
    if np.any(mask):
        colors.append(image_sampled[mask].mean(axis=0))

if -1 in labels_image:
    # 噪声点用黑色表示
    colors.append([0, 0, 0])

colors = np.array(colors)

# 分割图像（简化版：只对采样点）
if len(sample_indices) == len(image_array):
    segmented_image = colors[labels_image].reshape(h, w, d)
else:
    # 使用 K-Means 作为近似
    from sklearn.cluster import KMeans
    kmeans_seg = KMeans(n_clusters=n_colors, random_state=42)
    labels_full = kmeans_seg.fit_predict(image_array)
    segmented_image = kmeans_seg.cluster_centers_[labels_full].reshape(h, w, d)

plt.subplot(1, 3, 2)
plt.imshow(segmented_image)
plt.title(f'DBSCAN 分割 ({n_colors} 种颜色)')
plt.axis('off')

# 显示发现的颜色
plt.subplot(1, 3, 3)
for i, color in enumerate(colors):
    plt.axvspan(i, i+1, facecolor=color, alpha=0.7)
plt.title('发现的主要颜色')
plt.xlim(0, len(colors))
plt.ylim(0, 1)
plt.axis('off')

plt.tight_layout()
plt.show()

## 10. DBSCAN 的局限性

In [ ]:
# 演示 DBSCAN 的局限性

# 1. 对密度变化敏感
X_density_varied, y_density_varied, _ = datasets['sparse']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 密度变化的数据
axes[0, 0].scatter(X_density_varied[:, 0], X_density_varied[:, 1], 
                   c=y_density_varied, cmap='viridis', alpha=0.6)
axes[0, 0].set_xlabel('特征 1')
axes[0, 0].set_ylabel('特征 2')
axes[0, 0].set_title('密度变化的数据')
axes[0, 0].grid(True, alpha=0.3)

# 使用相同的 eps
dbscan_density = DBSCAN(eps=1.5, min_samples=5)
labels_density = dbscan_density.fit_predict(X_density_varied)

unique_labels = set(labels_density)
for k in unique_labels:
    class_member_mask = (labels_density == k)
    if k == -1:
        xy = X_density_varied[class_member_mask]
        axes[0, 1].scatter(xy[:, 0], xy[:, 1], c='black', marker='x', s=30, alpha=0.6)
    else:
        xy = X_density_varied[class_member_mask]
        axes[0, 1].scatter(xy[:, 0], xy[:, 1], alpha=0.6)
axes[0, 1].set_xlabel('特征 1')
axes[0, 1].set_ylabel('特征 2')
axes[0, 1].set_title('DBSCAN (统一 eps=1.5)')
axes[0, 1].grid(True, alpha=0.3)

# 2. 高维数据
from sklearn.datasets import make_classification
X_high_dim, y_high_dim = make_classification(
    n_samples=300, n_features=10, n_informative=5, 
    n_redundant=2, n_classes=3, random_state=42
)

# DBSCAN 在高维数据上
dbscan_high = DBSCAN(eps=5, min_samples=5)
labels_high = dbscan_high.fit_predict(X_high_dim)

print("DBSCAN 在高维数据上的表现:")
print(f"  发现簇数: {len(set(labels_high)) - (1 if -1 in labels_high else 0)}")
print(f"  噪声点数: {np.sum(labels_high == -1)}")
print(f"  真实类数: 3")

# 分析文本
limitations_text = """DBSCAN 的局限性:

1. 参数敏感:
   - eps 和 min_samples 需要仔细调优
   - 不同数据集需要不同参数

2. 密度变化:
   - 难以处理密度差异大的数据
   - 统一参数可能不适合所有簇

3. 高维数据:
   - 距离度量在高维空间失效
   - "维度灾难"问题

4. 边界区域:
   - 边界点可能被错误分类
   - 密度渐变区域效果差

解决方案:
- OPTICS: 自动调整 eps
- HDBSCAN: 层次化 DBSCAN
- 数据降维后聚类
"""
axes[0, 2].text(0.1, 0.1, limitations_text, fontsize=9, verticalalignment='bottom',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[0, 2].axis('off')

# 计算复杂度对比
sample_sizes = [100, 200, 400, 800]
times_dbscan = []
times_kmeans = []

print("\n计算复杂度对比:")
print("样本数\tDBSCAN时间\tK-Means时间")
print("-" * 50)

for n in sample_sizes:
    X_temp = np.random.randn(n, 2)
    
    # DBSCAN
    start = time.time()
    dbscan_temp = DBSCAN(eps=0.5, min_samples=5)
    dbscan_temp.fit_predict(X_temp)
    time_dbscan = time.time() - start
    times_dbscan.append(time_dbscan)
    
    # K-Means
    start = time.time()
    kmeans_temp = KMeans(n_clusters=3, random_state=42)
    kmeans_temp.fit_predict(X_temp)
    time_kmeans = time.time() - start
    times_kmeans.append(time_kmeans)
    
    print(f"{n}\t{time_dbscan:.4f}s\t\t{time_kmeans:.4f}s")

axes[1, 0].plot(sample_sizes, times_dbscan, 'b-o', label='DBSCAN', linewidth=2)
axes[1, 0].plot(sample_sizes, times_kmeans, 'r-o', label='K-Means', linewidth=2)
axes[1, 0].set_xlabel('样本数')
axes[1, 0].set_ylabel('运行时间 (秒)')
axes[1, 0].set_title('计算复杂度对比')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 参数建议
parameter_tips = """DBSCAN 参数选择建议:

1. 使用 K-距离图选择 eps:
   - 计算每个点到第 k 最近邻的距离
   - 找到曲线的肘部位置

2. min_samples 经验法则:
   - 最小值: 维度 + 1
   - 推荐: 维度 + 1 到 2*维度
   - 噪声多时增大该值

3. 特征标准化:
   - 重要！必须标准化
   - 使用 StandardScaler
   - 确保各特征尺度一致

4. 数据预处理:
   - 移除异常值（可选）
   - 降维（PCA, t-SNE）
   - 特征选择
"""
axes[1, 1].text(0.1, 0.1, parameter_tips, fontsize=9, verticalalignment='bottom',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
axes[1, 1].axis('off')

# 适用场景
use_cases = """DBSCAN 适用场景:

✓ 适合:
- 任意形状的簇
- 包含噪声的数据
- 密度相近的数据
- 异常检测
- 空间数据聚类

✗ 不适合:
- 密度差异大的数据
- 高维数据
- 需要指定簇数的场景
- 大规模数据（可考虑 HDBSCAN）

替代方案:
- OPTICS: 自动 eps
- HDBSCAN: 层次化
- K-Means: 大数据, 球形簇
- 层次聚类: 需要结构
"""
axes[1, 2].text(0.1, 0.1, use_cases, fontsize=9, verticalalignment='bottom',
               bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 总结

### DBSCAN 算法核心

1. **核心概念**
   - eps (ε): 邻域半径
   - min_samples: 最少邻域点数
   - 核心点、边界点、噪声点

2. **聚类过程**
   - 识别所有核心点
   - 通过密度可达性连接核心点
   - 将边界点分配到最近的核心点簇
   - 噪声点不属于任何簇

### 参数调优

| 参数 | 方法 | 建议 |
|------|------|------|
| eps | K-距离图 | 选择肘部位置 |
| min_samples | 经验法则 | 维度 + 1 到 2*维度 |

### DBSCAN vs 其他算法

选择 DBSCAN 当：
- 数据包含噪声
- 簇形状不规则
- 密度相近
- 不确定簇的数量

选择 K-Means 当：
- 大数据量
- 簇是球形的
- 需要指定簇数
- 计算效率重要

选择层次聚类当：
- 需要层次结构
- 数据量不大
- 需要可视化

### 实践建议

1. **数据预处理**
   - 必须标准化特征
   - 考虑降维
   - 处理异常值

2. **参数选择**
   - 使用 K-距离图
   - 从小到大测试 eps
   - 根据 min_samples 经验法则

3. **结果评估**
   - 检查噪声点比例
   - 观察簇的形状
   - 使用轮廓系数
   - 结合业务知识

4. **替代方案**
   - OPTICS: 自动 eps
   - HDBSCAN: 层次化 + 自动参数
   - 混合方法: 结合多种算法